# Module 7.3 — Corrective RAG (CRAG) with LangGraph

CRAG adds **self-correction** to the RAG loop:
1. Retrieve documents
2. **Grade** each document for relevance
3. If grading is POOR → fall back to **web search** + **rewrite query**
4. If grading is GOOD → generate answer

Built as a LangGraph state machine.

In [ ]:
# !pip install langgraph langchain-openai langchain-community tavily-python

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document
import operator

# ── State definition ──────────────────────────────────────────────────────────
class CRAGState(TypedDict):
    query          : str
    documents      : list[Document]
    grade          : str        # "relevant" | "irrelevant"
    final_answer   : str
    rewritten_query: str

llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    Document(page_content="Python is an interpreted, high-level, general-purpose programming language."),
    Document(page_content="LangChain is a framework for building applications powered by language models."),
    Document(page_content="Vector databases store and retrieve embeddings efficiently."),
]
vs = Chroma.from_documents(docs, embeddings, collection_name="crag_demo")

# ── Node functions ────────────────────────────────────────────────────────────
def retrieve(state: CRAGState) -> CRAGState:
    docs = vs.similarity_search(state["query"], k=2)
    return {**state, "documents": docs}

grade_prompt = ChatPromptTemplate.from_template("""
Is this document relevant to the query? Answer only 'relevant' or 'irrelevant'.
Query: {query}
Document: {document}
""")

def grade_documents(state: CRAGState) -> CRAGState:
    grades = []
    for doc in state["documents"]:
        result = (grade_prompt | llm | StrOutputParser()).invoke({
            "query": state["query"], "document": doc.page_content
        })
        grades.append(result.strip().lower())
    overall = "relevant" if any(g == "relevant" for g in grades) else "irrelevant"
    return {**state, "grade": overall}

def rewrite_query(state: CRAGState) -> CRAGState:
    rewrite = (
        ChatPromptTemplate.from_template("Rewrite this query for better web search: {query}")
        | llm | StrOutputParser()
    ).invoke({"query": state["query"]})
    return {**state, "rewritten_query": rewrite.strip()}

def web_search(state: CRAGState) -> CRAGState:
    # Simulated web search (replace with TavilySearch for real use)
    q   = state.get("rewritten_query", state["query"])
    sim = [Document(page_content=f"[Web result for: {q}] General information retrieved from the web.")]
    return {**state, "documents": sim}

gen_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.
Context: {context}
Question: {question}
""")

def generate(state: CRAGState) -> CRAGState:
    context = "\n".join(d.page_content for d in state["documents"])
    answer  = (gen_prompt | llm | StrOutputParser()).invoke({
        "context": context, "question": state["query"]
    })
    return {**state, "final_answer": answer}

def route_grade(state: CRAGState) -> str:
    return "generate" if state["grade"] == "relevant" else "rewrite_query"

# ── Build LangGraph ───────────────────────────────────────────────────────────
graph = StateGraph(CRAGState)
graph.add_node("retrieve",       retrieve)
graph.add_node("grade",          grade_documents)
graph.add_node("rewrite_query",  rewrite_query)
graph.add_node("web_search",     web_search)
graph.add_node("generate",       generate)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "grade")
graph.add_conditional_edges("grade", route_grade, {"generate": "generate", "rewrite_query": "rewrite_query"})
graph.add_edge("rewrite_query", "web_search")
graph.add_edge("web_search",    "generate")
graph.add_edge("generate",      END)

crag = graph.compile()

# ── Run ───────────────────────────────────────────────────────────────────────
for query in ["What is LangChain?", "Who invented the steam engine?"]:
    result = crag.invoke({"query": query, "documents": [], "grade": "",
                          "final_answer": "", "rewritten_query": ""})
    print(f"Query  : {query}")
    print(f"Grade  : {result['grade']}")
    print(f"Answer : {result['final_answer']}\n{'─'*60}\n")
